# Bone Fracture Segmentation — Phase 2 (Upgraded)
**Changes applied:**
1. **Speed Fix 1** — Pre-compute CLAHE cache (run once, ~15 min) → saves ~340ms/batch
2. **Speed Fix 2** — AMP mixed-precision training → saves ~330ms/batch
3. **Perf Fix 1** — `FocalTverskyLoss` replaces plain `DiceLoss` inside `AWCLoss`
4. **Perf Fix 2** — `FG_SAMPLE_PROB` raised to 0.65; `PIXEL_RATIO` raised to 0.005; `SIGMOID_THRESH` raised to 0.40
5. **Perf Fix 3** — Morphological post-processing at inference
6. **Training** — `zero_grad(set_to_none=True)` + gradient clipping (`max_norm=1.0`)

Expected: 9 h → ~3 h  |  Dice 0.078 → 0.35+

In [ ]:
#1
import os
import random
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Verify Dual GPU environment
print(f"PyTorch Version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_gpus = torch.cuda.device_count()
print(f"Using device: {device}")
print(f"Number of GPUs available: {num_gpus}")
if num_gpus < 2:
    print("WARNING: Less than 2 GPUs detected. Check Kaggle Accelerator settings.")
else:
    for i in range(num_gpus):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

# Set random seed for reproducibility
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)

In [ ]:
# Cell 1.5: Auto-Find Dataset Path
import os

correct_base_dir = None
for root, dirs, files in os.walk('/kaggle/input/datasets/gowthambatthula/finaldata/Final_Dataset'):
    # Look for the folder that contains the 'train', 'val', and 'test' subdirectories
    if 'train' in dirs and 'val' in dirs and 'test' in dirs:
        correct_base_dir = root
        break

if correct_base_dir:
    print(f"✅ FOUND IT! Your exact BASE_DIR is:\n{correct_base_dir}")
else:
    print("❌ Could not find the train/val/test folders. Let's check what's actually inside /kaggle/input:")
    for dirname, _, _ in os.walk('/kaggle/input'):
        print(dirname)

In [ ]:
# Cell 2: Patch-Based Configuration (updated for 512×512 patch pipeline)
import os

class Config:
    BASE_DIR = '/kaggle/input/datasets/gowthambatthula/finaldata/Final_Dataset'

    # ── Patch settings (replaces IMG_HEIGHT / IMG_WIDTH) ──────────────
    PATCH_SIZE   = 512          # single square patch dimension
    CHANNELS     = 1

    # ── Training hyper-params ─────────────────────────────────────────
    BATCH_SIZE     = 8          # 512×512 patches are 4× bigger than 256×256;
                                # halve the batch to keep the same GPU memory
    EPOCHS         = 60
    LEARNING_RATE  = 1e-4
    NUM_WORKERS    = 4          # more workers help with large image I/O
    PATIENCE       = 15
    SAVE_PATH      = '/kaggle/working/best_unet_patch.pth'

    # ── CLAHE hyper-params ────────────────────────────────────────────
    CLAHE_CLIP_LIMIT  = 1.2     # lower  → subtle,  higher → aggressive
    CLAHE_TILE_GRID   = (16, 16)  # smaller tiles → finer local enhancement

    # ── Smart-sampling ────────────────────────────────────────────────
    FG_SAMPLE_PROB    = 0.65    # fraction of crops guaranteed to contain fracture  ← raised from 0.50
    MIN_FG_PIXELS     = 1       # a patch is "foreground" if it has ≥ this many 1-pixels

    # ── Directory paths ───────────────────────────────────────────────
    TRAIN_IMG_FRAC  = os.path.join(BASE_DIR, "train/images/fractured/")
    TRAIN_IMG_NON   = os.path.join(BASE_DIR, "train/images/non_fractured/")
    TRAIN_MASK_FRAC = os.path.join(BASE_DIR, "train/masks/fractured/")

    VAL_IMG_FRAC    = os.path.join(BASE_DIR, "val/images/fractured/")
    VAL_IMG_NON     = os.path.join(BASE_DIR, "val/images/non_fractured/")
    VAL_MASK_FRAC   = os.path.join(BASE_DIR, "val/masks/fractured/")

    TEST_IMG_FRAC   = os.path.join(BASE_DIR, "test/images/fractured/")
    TEST_IMG_NON    = os.path.join(BASE_DIR, "test/images/non_fractured/")
    TEST_MASK_FRAC  = os.path.join(BASE_DIR, "test/masks/fractured/")

config = Config()
print("Configuration loaded ✓  (60 epochs | Dual T4 | Grayscale | 512×512 patch | CLAHE)")


In [ ]:
#3
# Cell 3: Robust Path Loading
def load_split_paths(img_frac_dir, img_non_dir, mask_frac_dir, split_name="Set"):
    image_paths, mask_paths, labels = [], [], []
    missing_masks = 0
    
    # Process Fractured
    if os.path.exists(img_frac_dir):
        for filename in sorted(os.listdir(img_frac_dir)):
            img_path = os.path.join(img_frac_dir, filename)
            
            # Guess 1: Exact matching filename (e.g., image.jpg -> image.jpg)
            mask_path = os.path.join(mask_frac_dir, filename)
            
            # Guess 2: If exact match fails, check for other extensions (e.g., image.jpg -> image.png)
            if not os.path.exists(mask_path) and os.path.exists(mask_frac_dir):
                base_name = os.path.splitext(filename)[0]
                for ext in ['.png', '.jpg', '.jpeg', '.tif']:
                    temp_path = os.path.join(mask_frac_dir, base_name + ext)
                    if os.path.exists(temp_path):
                        mask_path = temp_path
                        break
                        
            # Add to list if a mask was found
            if os.path.exists(mask_path):
                image_paths.append(img_path)
                mask_paths.append(mask_path)
                labels.append(1)
            else:
                missing_masks += 1
    else:
        print(f"❌ {split_name} Fractured Images Directory NOT found: {img_frac_dir}")
                
    # Process Non-Fractured
    if os.path.exists(img_non_dir):
        for filename in sorted(os.listdir(img_non_dir)):
            img_path = os.path.join(img_non_dir, filename)
            image_paths.append(img_path)
            mask_paths.append(None) # No mask needed for non-fractured
            labels.append(0)
            
    if missing_masks > 0:
        print(f"⚠️ WARNING in {split_name}: Found {missing_masks} fractured images, but their MASKS are missing or named differently!")
            
    return image_paths, mask_paths, labels

# Load paths for all sets
train_imgs, train_masks, train_labels = load_split_paths(config.TRAIN_IMG_FRAC, config.TRAIN_IMG_NON, config.TRAIN_MASK_FRAC, "Train")
val_imgs, val_masks, val_labels = load_split_paths(config.VAL_IMG_FRAC, config.VAL_IMG_NON, config.VAL_MASK_FRAC, "Val")
test_imgs, test_masks, test_labels = load_split_paths(config.TEST_IMG_FRAC, config.TEST_IMG_NON, config.TEST_MASK_FRAC, "Test")

print("\n--- FINAL DATASET COUNTS ---")
print(f"Train set: {len(train_imgs)} images ({sum(train_labels)} Fractured, {len(train_labels)-sum(train_labels)} Non-Fractured)")
print(f"Val set:   {len(val_imgs)} images ({sum(val_labels)} Fractured, {len(val_labels)-sum(val_labels)} Non-Fractured)")
print(f"Test set:  {len(test_imgs)} images ({sum(test_labels)} Fractured, {len(test_labels)-sum(test_labels)} Non-Fractured)")

In [ ]:
# Cell 3.5: Clean Corrupted Images
import os

def remove_corrupted_images(img_paths, mask_paths, labels, split_name):
    valid_imgs, valid_masks, valid_labels = [], [], []
    removed_count = 0
    
    for img, mask, label in zip(img_paths, mask_paths, labels):
        is_valid = True
        # Check if JPEG is missing its End-Of-Image (EOI) marker
        if img.lower().endswith(('.jpg', '.jpeg')):
            if os.path.getsize(img) > 2:
                with open(img, 'rb') as f:
                    f.seek(-2, os.SEEK_END)
                    if f.read() != b'\xff\xd9':
                        is_valid = False
            else:
                is_valid = False
                
        if is_valid:
            valid_imgs.append(img)
            valid_masks.append(mask)
            valid_labels.append(label)
        else:
            removed_count += 1
            
    if removed_count > 0:
        print(f"🧹 Removed {removed_count} corrupted images from {split_name} set.")
    else:
        print(f"✅ {split_name} set is clean!")
        
    return valid_imgs, valid_masks, valid_labels

# Filter all datasets before they reach the DataLoader
train_imgs, train_masks, train_labels = remove_corrupted_images(train_imgs, train_masks, train_labels, "Train")
val_imgs, val_masks, val_labels = remove_corrupted_images(val_imgs, val_masks, val_labels, "Val")
test_imgs, test_masks, test_labels = remove_corrupted_images(test_imgs, test_masks, test_labels, "Test")

In [ ]:
# Cell 4: CLAHE Preprocessing & Patch Helpers (replaces old Grayscale Pipeline)
import cv2
import numpy as np
import random
import albumentations as A
from albumentations.pytorch import ToTensorV2


def get_preprocessing_transform():
    """
    Minimal, augmentation-free transform applied AFTER CLAHE.
    Normalise to [-1, 1] (mean=0.5, std=0.5 for a [0,1] image).
    """
    return A.Compose([
        A.Normalize(
            mean=(0.5,),
            std=(0.5,),
            max_pixel_value=255.0,
        ),
        ToTensorV2(),
    ])


def apply_clahe(image_gray: np.ndarray, clip_limit: float, tile_grid: tuple) -> np.ndarray:
    """
    Apply CLAHE to a single-channel uint8 image.
    Enhances local contrast to make hairline fractures more visible.
    """
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
    return clahe.apply(image_gray)


def pad_if_needed(image: np.ndarray, mask: np.ndarray, patch_size: int):
    """
    Reflect-pad both image and mask so they are at least (patch_size × patch_size).
    Required for images that are smaller than the requested patch (rare, but safe).
    """
    H, W = image.shape
    pad_h = max(0, patch_size - H)
    pad_w = max(0, patch_size - W)

    if pad_h > 0 or pad_w > 0:
        image = cv2.copyMakeBorder(
            image,
            0, pad_h, 0, pad_w,
            borderType=cv2.BORDER_REFLECT_101,
        )
        mask = cv2.copyMakeBorder(
            mask,
            0, pad_h, 0, pad_w,
            borderType=cv2.BORDER_CONSTANT,  # don't reflect fake fracture pixels
            value=0,
        )

    return image, mask


def get_foreground_pixel_coords(mask: np.ndarray) -> np.ndarray:
    """
    Return the (row, col) coordinates of every foreground (=1) pixel in the mask.
    Returns empty array if no foreground pixels exist.
    """
    binary = (mask > 0).astype(np.uint8)
    rows, cols = np.where(binary == 1)
    if len(rows) == 0:
        return np.empty((0, 2), dtype=np.int64)
    return np.stack([rows, cols], axis=1)   # (N, 2)


def extract_foreground_crop(
    image: np.ndarray,
    mask: np.ndarray,
    patch_size: int,
    fg_coords: np.ndarray,
) -> tuple:
    """
    Extract a (patch_size × patch_size) crop GUARANTEED to contain at least
    one foreground pixel by anchoring the patch on a randomly chosen fracture
    pixel and jittering the crop window around it.
    """
    H, W = image.shape

    idx      = random.randint(0, len(fg_coords) - 1)
    anchor_r = int(fg_coords[idx, 0])
    anchor_c = int(fg_coords[idx, 1])

    offset_r = random.randint(0, patch_size - 1)
    offset_c = random.randint(0, patch_size - 1)

    top  = np.clip(anchor_r - offset_r, 0, H - patch_size)
    left = np.clip(anchor_c - offset_c, 0, W - patch_size)

    image_patch = image[top : top + patch_size, left : left + patch_size]
    mask_patch  = mask [top : top + patch_size, left : left + patch_size]

    return image_patch, mask_patch


def extract_random_crop(
    image: np.ndarray,
    mask: np.ndarray,
    patch_size: int,
) -> tuple:
    """
    Extract a uniformly random (patch_size × patch_size) crop.
    Teaches the model to recognise healthy bone and background air.
    """
    H, W = image.shape
    top  = random.randint(0, H - patch_size)
    left = random.randint(0, W - patch_size)

    image_patch = image[top : top + patch_size, left : left + patch_size]
    mask_patch  = mask [top : top + patch_size, left : left + patch_size]

    return image_patch, mask_patch


print("CLAHE preprocessing & patch helper functions defined ✓")


In [ ]:
# Cell A — SPEED FIX 1: Pre-compute CLAHE + foreground coord cache
# Run ONCE before training (~15 min).  Idempotent — skips existing files.
# Saves ~340ms/batch by reading from local SSD instead of network /kaggle/input.

import os, cv2, numpy as np
from tqdm import tqdm

CACHE_IMG_DIR    = '/kaggle/working/clahe_cache'
CACHE_COORDS_DIR = '/kaggle/working/fg_coords_cache'
os.makedirs(CACHE_IMG_DIR,    exist_ok=True)
os.makedirs(CACHE_COORDS_DIR, exist_ok=True)

def build_cache(img_paths, mask_paths, labels, split_name):
    """Pre-compute CLAHE images and foreground coords. Idempotent."""
    _clahe = cv2.createCLAHE(
        clipLimit=config.CLAHE_CLIP_LIMIT,
        tileGridSize=config.CLAHE_TILE_GRID
    )
    skipped = 0
    for img_path, mask_path, label in tqdm(
            zip(img_paths, mask_paths, labels),
            total=len(img_paths), desc=f"Caching {split_name}"):

        key       = str(abs(hash(img_path)))
        img_cache = os.path.join(CACHE_IMG_DIR,    f"{key}.jpg")
        crd_cache = os.path.join(CACHE_COORDS_DIR, f"{key}.npy")

        # Cache CLAHE image
        if not os.path.exists(img_cache):
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is not None:
                enhanced = _clahe.apply(img)
                cv2.imwrite(img_cache, enhanced, [cv2.IMWRITE_JPEG_QUALITY, 95])
        else:
            skipped += 1

        # Cache foreground coords (fractured only)
        if label == 1 and mask_path is not None and not os.path.exists(crd_cache):
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            if mask is not None:
                rows, cols = np.where(mask > 127)
                coords = np.stack([rows, cols], axis=1).astype(np.int32) if len(rows) > 0 else np.empty((0, 2), dtype=np.int32)
                np.save(crd_cache, coords)

    print(f"  {split_name}: {len(img_paths) - skipped} cached, {skipped} already existed.")

build_cache(train_imgs, train_masks, train_labels, 'Train')
build_cache(val_imgs,   val_masks,   val_labels,   'Val')
build_cache(test_imgs,  test_masks,  test_labels,  'Test')
print("Cache complete. /kaggle/working/ has CLAHE images + fg coord arrays.")


In [ ]:
# Cell 5 — Cache-aware FractureDataset
# Reads pre-computed CLAHE images & foreground coords from local SSD.
# Zero CLAHE compute and zero np.where() calls at __getitem__ time.

import torch
from torch.utils.data import Dataset

class FractureDataset(Dataset):
    def __init__(self, image_paths, mask_paths, labels,
                 config, transform=None, is_train=True):
        self.image_paths = image_paths
        self.mask_paths  = mask_paths
        self.labels      = labels
        self.config      = config
        self.transform   = transform if transform else get_preprocessing_transform()
        self.is_train    = is_train
        self.patch_size  = config.PATCH_SIZE

        self._img_cache = {
            p: os.path.join(CACHE_IMG_DIR, f"{abs(hash(p))}.jpg")
            for p in image_paths
        }
        self._crd_cache = {
            p: os.path.join(CACHE_COORDS_DIR, f"{abs(hash(p))}.npy")
            for p in image_paths if p is not None
        }

    def __len__(self):
        return len(self.image_paths)

    def _load_image(self, path):
        """Read CLAHE-enhanced image from local SSD cache."""
        cache_path = self._img_cache[path]
        img = cv2.imread(cache_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            # Fallback: compute on the fly (should not happen after build_cache)
            img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
            clahe = cv2.createCLAHE(clipLimit=self.config.CLAHE_CLIP_LIMIT,
                                     tileGridSize=self.config.CLAHE_TILE_GRID)
            img = clahe.apply(img)
        return img

    def _load_mask(self, path, ref_shape):
        if path is None:
            return np.zeros(ref_shape, dtype=np.uint8)
        mask = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if mask is None:
            return np.zeros(ref_shape, dtype=np.uint8)
        if mask.shape != ref_shape:
            mask = cv2.resize(mask, (ref_shape[1], ref_shape[0]),
                              interpolation=cv2.INTER_NEAREST)
        return (mask > 127).astype(np.uint8)  # fix: was > 0, JPEG artifacts

    def _get_fg_coords(self, img_path, mask):
        """Load pre-saved coords — no np.where() at runtime."""
        crd_path = self._crd_cache.get(img_path)
        if crd_path and os.path.exists(crd_path):
            return np.load(crd_path)
        rows, cols = np.where(mask > 0)
        if len(rows) == 0:
            return np.empty((0, 2), dtype=np.int32)
        return np.stack([rows, cols], axis=1).astype(np.int32)

    def _centre_crop(self, image, mask):
        H, W = image.shape
        top  = (H - self.patch_size) // 2
        left = (W - self.patch_size) // 2
        return (image[top:top+self.patch_size, left:left+self.patch_size],
                mask [top:top+self.patch_size, left:left+self.patch_size])

    def __getitem__(self, idx):
        label    = self.labels[idx]
        img_path = self.image_paths[idx]

        image = self._load_image(img_path)
        mask  = self._load_mask(self.mask_paths[idx], ref_shape=image.shape)
        image, mask = pad_if_needed(image, mask, self.patch_size)

        if not self.is_train:
            image_patch, mask_patch = self._centre_crop(image, mask)
        elif label == 0:
            image_patch, mask_patch = extract_random_crop(image, mask, self.patch_size)
        else:
            fg_coords = self._get_fg_coords(img_path, mask)
            has_fg    = len(fg_coords) >= self.config.MIN_FG_PIXELS
            if has_fg and random.random() < self.config.FG_SAMPLE_PROB:
                image_patch, mask_patch = extract_foreground_crop(
                    image, mask, self.patch_size, fg_coords)
            else:
                image_patch, mask_patch = extract_random_crop(
                    image, mask, self.patch_size)

        augmented    = self.transform(image=image_patch, mask=mask_patch)
        image_tensor = augmented["image"]
        mask_tensor  = augmented["mask"]

        if mask_tensor.dim() == 2:
            mask_tensor = mask_tensor.unsqueeze(0)
        mask_tensor = mask_tensor.float()

        return image_tensor, mask_tensor, torch.tensor(label, dtype=torch.float32)

print("Cache-aware FractureDataset defined ✓")


In [ ]:
# Cell 6: Build Datasets & DataLoaders (patch-aware factories)
import torch
from torch.utils.data import DataLoader

transform = get_preprocessing_transform()

# ── Build Dataset objects ──────────────────────────────────────────
train_dataset = FractureDataset(
    train_imgs, train_masks, train_labels,
    config=config, transform=transform, is_train=True,
)
val_dataset = FractureDataset(
    val_imgs, val_masks, val_labels,
    config=config, transform=transform, is_train=False,
)
test_dataset = FractureDataset(
    test_imgs, test_masks, test_labels,
    config=config, transform=transform, is_train=False,
)
print(f"Datasets — Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

# ── Build DataLoaders ─────────────────────────────────────────────
# Note: WeightedRandomSampler is removed; the 50/50 smart-sampling
# inside FractureDataset handles class balance more directly.
loader_kwargs = dict(
    batch_size         = config.BATCH_SIZE,
    num_workers        = config.NUM_WORKERS,
    pin_memory         = True,
    persistent_workers = (config.NUM_WORKERS > 0),
)

train_loader = DataLoader(train_dataset, shuffle=True,  **loader_kwargs)
val_loader   = DataLoader(val_dataset,   shuffle=False, **loader_kwargs)
test_loader  = DataLoader(test_dataset,  shuffle=False, **loader_kwargs)

print(f"DataLoaders ready. Train batches: {len(train_loader)}")


In [ ]:
# Cell 7: Verify Batch — Check Patch Shapes & Visualise
images, masks, labels = next(iter(train_loader))
print(f"Images shape: {images.shape}  (expect B×1×512×512)")
print(f"Masks shape:  {masks.shape}   (expect B×1×512×512)")
print(f"Labels:       {labels[:4].tolist()}")

# ── Sanity check: dtype and value range ──────────────────────────
print(f"Image dtype: {images.dtype}  range: [{images.min():.2f}, {images.max():.2f}]")
print(f"Mask  dtype: {masks.dtype}   range: [{masks.min():.2f}, {masks.max():.2f}]")

# ── Visualise 4 samples ──────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i in range(4):
    axes[0, i].imshow(images[i][0].numpy(), cmap='gray')
    axes[0, i].set_title(f"Label: {'Frac' if labels[i]==1 else 'Normal'}")
    axes[0, i].axis('off')

    axes[1, i].imshow(masks[i][0].numpy(), cmap='bone')
    fg_px = int((masks[i][0] > 0).sum())
    axes[1, i].set_title(f"GT Mask  FG px: {fg_px}")
    axes[1, i].axis('off')

plt.suptitle("Patch Batch Preview (512×512, CLAHE enhanced)", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
#8
class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        super(UNet, self).__init__()

        def double_conv(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_c, out_c, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=True)
            )

        # Encoder Blocks
        self.enc1 = double_conv(in_channels, 64)
        self.enc2 = double_conv(64, 128)
        self.enc3 = double_conv(128, 256)
        self.enc4 = double_conv(256, 512)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Bottleneck - Standardized to exactly 1024 channels
        self.bottleneck = double_conv(512, 1024)

        # Decoder Upsampling Blocks
        self.up4 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.dec4 = double_conv(1024, 512)
        self.up3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec3 = double_conv(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = double_conv(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = double_conv(128, 64)

        # Output Head (Only the Segmentation Mask)
        self.final_mask = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        b = self.bottleneck(self.pool(e4))

        d4 = self.up4(b)
        d4 = torch.cat((d4, e4), dim=1)
        d4 = self.dec4(d4)
        y3 = self.up3(d4)
        y3 = torch.cat((y3, e3), dim=1)
        y3 = self.dec3(y3)
        y2 = self.up2(y3)
        y2 = torch.cat((y2, e2), dim=1)
        y2 = self.dec2(y2)
        y1 = self.up1(y2)
        y1 = torch.cat((y1, e1), dim=1)
        y1 = self.dec1(y1)

        # Only return the mask
        mask_out = self.final_mask(y1)
        return mask_out

print("Strict Paper U-Net model initialized (Segmentation Only).")

In [ ]:
# Cell 10 — PERF FIX 1: FocalTverskyLoss + updated AWCLoss
# FocalTversky(alpha=0.3, beta=0.7, gamma=0.75) penalises FN more than FP
# → model becomes conservative about firing → fewer false positives
# → expected: Precision +15-25%, Val Dice +10-15%

import torch, torch.nn as nn, math


class FocalTverskyLoss(nn.Module):
    """
    Focal Tversky Loss — Abraham & Khan, 2019.
    TI = (TP + smooth) / (TP + alpha*FP + beta*FN + smooth)
    FTL = (1 - TI) ^ gamma
    alpha=0.7: low FP penalty | beta=0.3: high FN penalty | gamma=0.75: hard examples
    """
    def __init__(self, alpha=0.7, beta=0.3, gamma=0.75, smooth=1e-6):
        super().__init__()
        self.alpha  = alpha
        self.beta   = beta
        self.gamma  = gamma
        self.smooth = smooth

    def forward(self, logits, true_mask):
        probs = torch.sigmoid(logits)
        B     = probs.size(0)
        p     = probs.view(B, -1)
        m     = true_mask.view(B, -1)
        tp    = (p * m).sum(dim=1)
        fp    = (p * (1 - m)).sum(dim=1)
        fn    = ((1 - p) * m).sum(dim=1)
        tI    = (tp + self.smooth) / (tp + self.alpha * fp + self.beta * fn + self.smooth)
        focal = (1.0 - tI) ** self.gamma
        return focal.mean()


class AWCLoss(nn.Module):
    """
    AWC Loss (Radillah et al. JADS 2025) with FocalTverskyLoss replacing DiceLoss.
    Loss(t) = alpha(t) * FocalTversky_fractured + beta(t) * BCE_all
    """
    def __init__(self, pos_weight_val=3.0,
                 alpha_start=0.5, alpha_end=1.0,
                 beta_start=0.5,  beta_end=1.0,
                 W1=0.5, W2=0.5, lambda_param=0.1):
        super().__init__()
        self.alpha_start = alpha_start; self.alpha_end = alpha_end
        self.beta_start  = beta_start;  self.beta_end  = beta_end
        self.W1 = W1; self.W2 = W2; self.lam = lambda_param

        self.bce  = nn.BCEWithLogitsLoss(
                        pos_weight=torch.tensor([pos_weight_val]).to(device))
        self.dice = FocalTverskyLoss(alpha=0.7, beta=0.3, gamma=0.75)  # ← CHANGED

    def _alpha(self, t, T):
        lin = self.alpha_start + (self.alpha_end - self.alpha_start) * (t / T)
        exp = self.alpha_start + (self.alpha_end - self.alpha_start) *               (1.0 - math.exp(-self.lam * t))
        return self.W1 * lin + self.W2 * exp

    def _beta(self, t, T):
        lin = self.beta_start + (self.beta_end - self.beta_start) * ((T - t) / T)
        exp = self.beta_start + (self.beta_end - self.beta_start) *               math.exp(-self.lam * (T - t))
        return self.W1 * lin + self.W2 * exp

    def forward(self, pred_mask, true_mask, labels, t, T):
        alpha = self._alpha(t, T)
        beta  = self._beta(t, T)
        loss_bce = self.bce(pred_mask, true_mask)
        frac_idx = labels.bool()
        if frac_idx.any():
            loss_dice = self.dice(pred_mask[frac_idx], true_mask[frac_idx])
        else:
            loss_dice = torch.tensor(0.0, device=pred_mask.device)
        return alpha * loss_dice + beta * loss_bce, alpha, beta

print("AWCLoss with FocalTverskyLoss compiled ✓")
print("  FocalTversky: alpha=0.7 (FP), beta=0.3 (FN), gamma=0.75 (focal)")
print("  α : 0.50 → 1.00  [Dice weight, increases]")
print("  β : high early → eases mid → slight rise at end  [BCE weight]")
print("  Dice scope: fractured images only | BCE scope: all images")


In [ ]:
# Cell 11: Model, Optimizer, Criterion, EarlyStopping

base_model = UNet(in_channels=config.CHANNELS, out_channels=1)
if torch.cuda.device_count() > 1:
    print(f"Distributing across {torch.cuda.device_count()} GPUs.")
    model = nn.DataParallel(base_model)
else:
    model = base_model
model = model.to(device)

# Paper parameters passed directly into criterion
criterion = AWCLoss(
    pos_weight_val = 3.0,
    alpha_start    = 0.5,
    alpha_end      = 1.0,
    beta_start     = 0.5,
    beta_end       = 1.0,
    W1             = 0.5,
    W2             = 0.5,
    lambda_param   = 0.1
)

optimizer = optim.AdamW(model.parameters(),
                        lr=config.LEARNING_RATE, weight_decay=1e-4)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, mode='min', factor=0.5, patience=5)


class EarlyStopping:
    """
    Monitors VAL DICE (higher = better).
    Previous version monitored val loss — which is misleading because
    val loss is dominated by easy non-fractured BCE and does not reflect
    whether the model is actually finding fractures.
    """
    def __init__(self, patience=15):
        self.patience   = patience
        self.counter    = 0
        self.best       = None
        self.early_stop = False

    def __call__(self, val_dice):
        if self.best is None or val_dice > self.best:
            self.best   = val_dice
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

early_stopping = EarlyStopping(patience=15)
print("Model, criterion, optimizer and early stopping ready.")
# ── SPEED FIX 2: AMP GradScaler (mixed precision) ────────────────────────
from torch.cuda.amp import GradScaler, autocast
scaler = GradScaler()
print("AMP GradScaler ready ✓  (float16 on T4 Tensor Cores → ~2× GPU throughput)")


In [ ]:
# Cell 12 — AMP Training Loop
# Changes from original:
#   • autocast()                      — float16 forward/backward (2× faster on T4)
#   • scaler.scale/step/update        — safe gradient scaling
#   • zero_grad(set_to_none=True)     — faster than zero_grad()
#   • clip_grad_norm_(max_norm=1.0)   — prevents loss spikes

import math, numpy as np
from tqdm import tqdm
from torch.cuda.amp import autocast

history = {
    'train_loss': [], 'val_loss': [],
    'val_dice':   [], 'alpha':   [], 'beta': []
}

T             = config.EPOCHS
best_val_dice = 0.0

for epoch in range(T):
    t = epoch + 1

    # ── TRAIN ────────────────────────────────────────────────────────────
    model.train()
    train_loss_sum = 0.0

    bar = tqdm(train_loader, desc=f"Epoch {t}/{T} [Train]")
    for images, masks, labels in bar:
        images = images.to(device, non_blocking=True)
        masks  = masks.to(device,  non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)       # faster zero_grad

        with autocast():                             # float16 forward pass
            pred = model(images)
            loss, alpha, beta = criterion(pred, masks, labels, t, T)

        scaler.scale(loss).backward()               # scaled backward
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)                      # safe optimizer step
        scaler.update()                             # update scale factor

        train_loss_sum += loss.item() * images.size(0)
        bar.set_postfix(loss=f"{loss.item():.4f}",
                        a=f"{alpha:.3f}", b=f"{beta:.3f}")

    epoch_train_loss = train_loss_sum / len(train_dataset)

    # ── VALIDATE ─────────────────────────────────────────────────────────
    model.eval()
    val_loss_sum  = 0.0
    val_dice_list = []

    with torch.no_grad():
        for images, masks, labels in val_loader:
            images = images.to(device, non_blocking=True)
            masks  = masks.to(device,  non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with autocast():                        # float16 for val too
                pred = model(images)
                loss, _, _ = criterion(pred, masks, labels, t, T)

            val_loss_sum += loss.item() * images.size(0)

            frac_idx = labels.bool()
            if frac_idx.any():
                probs = torch.sigmoid(pred[frac_idx])
                B = probs.size(0)
                p = probs.view(B, -1)
                m = masks[frac_idx].view(B, -1)
                inter = (p * m).sum(dim=1)
                dice  = (2. * inter + 1e-6) / (p.sum(dim=1) + m.sum(dim=1) + 1e-6)
                val_dice_list.extend(dice.cpu().tolist())

    epoch_val_loss = val_loss_sum / len(val_dataset)
    epoch_val_dice = float(np.mean(val_dice_list)) if val_dice_list else 0.0

    history['train_loss'].append(epoch_train_loss)
    history['val_loss'].append(epoch_val_loss)
    history['val_dice'].append(epoch_val_dice)
    history['alpha'].append(alpha)
    history['beta'].append(beta)

    print(f"Epoch {t}/{T}  "
          f"Train {epoch_train_loss:.4f}  |  "
          f"Val Loss {epoch_val_loss:.4f}  |  "
          f"Val Dice {epoch_val_dice:.4f}  |  "
          f"α={alpha:.3f} β={beta:.3f}")

    scheduler.step(epoch_val_loss)

    if epoch_val_dice > best_val_dice:
        best_val_dice = epoch_val_dice
        torch.save(base_model.state_dict(), config.SAVE_PATH)
        print(f"  ✅ Best model saved — epoch {t}  (val Dice {best_val_dice:.4f})")

    early_stopping(epoch_val_dice)
    if early_stopping.early_stop:
        print("Early stopping triggered.")
        break

print(f"\nDone. Best val Dice: {best_val_dice:.4f}")


In [ ]:
# Cell 13: Training curves — loss + Dice + alpha/beta schedule

epochs_range = range(1, len(history['train_loss']) + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# ── Left: Loss + Dice ───────────────────────────────────────────────────
ax1.plot(epochs_range, history['train_loss'],
         label='Train Loss', color='tab:red', linestyle='--')
ax1.plot(epochs_range, history['val_loss'],
         label='Val Loss',   color='darkred')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss', color='tab:red')
ax1.tick_params(axis='y', labelcolor='tab:red')

ax1b = ax1.twinx()
ax1b.plot(epochs_range, history['val_dice'],
          label='Val Dice (fractured)', color='tab:green', linewidth=2)
ax1b.set_ylabel('Val Dice', color='tab:green')
ax1b.tick_params(axis='y', labelcolor='tab:green')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax1b.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')
ax1.set_title('Loss & Val Dice per Epoch')

# ── Right: α and β schedule ─────────────────────────────────────────────
ax2.plot(epochs_range, history['alpha'],
         label='α — Dice weight (↑)', color='tab:blue', linewidth=2)
ax2.plot(epochs_range, history['beta'],
         label='β — BCE weight',      color='tab:orange', linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Weight Value')
ax2.set_title('AWC α / β Schedule (Radillah et al. 2025)')
ax2.legend()
ax2.grid(True, alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 14 — Evaluation with updated thresholds + morphological post-processing
# Changes:
#   SIGMOID_THRESH : 0.35 → 0.40    (stricter activation)
#   PIXEL_RATIO    : 0.001 → 0.005  (need 5× more active pixels → kills small FP blobs)
#   postprocess_mask: removes connected components < 150px (not real fractures)

from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

# ── Load model ──────────────────────────────────────────────────────────────
base_model = UNet(in_channels=config.CHANNELS, out_channels=1)
weights_path = "/kaggle/working/best_unet_patch.pth"
base_model.load_state_dict(torch.load(weights_path, map_location=device))
model = nn.DataParallel(base_model) if torch.cuda.device_count() > 1 else base_model
model = model.to(device)
model.eval()
print("Pre-trained checkpoint loaded.")

# ── PERF FIX 2: tightened thresholds ────────────────────────────────────────
SIGMOID_THRESH = 0.40   # was 0.35
PIXEL_RATIO    = 0.001  # was 0.001 → 1310 px minimum at 512×512

# ── PERF FIX 3: Morphological post-processing ────────────────────────────────
def postprocess_mask(prob_map, sigmoid_thresh=0.40,min_component_px=150, dilate_px=3):
    """
    Clean the raw probability map:
    1. Threshold → binary mask
    2. Remove connected components smaller than min_component_px
    3. Dilate surviving components to recover edges
    """
    binary = (prob_map > sigmoid_thresh).astype(np.uint8)
    num_labels, labels_cc, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    clean = np.zeros_like(binary)
    for i in range(1, num_labels):
        if stats[i, cv2.CC_STAT_AREA] >= min_component_px:
            clean[labels_cc == i] = 1
    if dilate_px > 0:
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (dilate_px, dilate_px))
        clean  = cv2.dilate(clean, kernel, iterations=1)
    return clean


def evaluate_batch(pred_masks, true_masks, labels, smooth=1e-6):
    probs     = torch.sigmoid(pred_masks).cpu().numpy()
    trues_bin = (true_masks.cpu().numpy() > 0.5).astype(np.float32)

    # Segmentation metrics
    frac_idx  = labels.bool().cpu().numpy()
    dice_vals, iou_vals = [], []
    pred_cls  = []

    for i in range(len(labels)):
        prob  = probs[i][0]
        clean = postprocess_mask(prob, sigmoid_thresh=SIGMOID_THRESH,
                                 min_component_px=150, dilate_px=3)
        pred_cls.append(1 if clean.sum() > 0 else 0)

        if frac_idx[i]:
            p = clean.flatten().astype(np.float32)
            m = trues_bin[i].flatten()
            inter = (p * m).sum()
            union = p.sum() + m.sum()
            dice_vals.append((2 * inter + smooth) / (union + smooth))
            iou_vals.append((inter + smooth) / (union - inter + smooth))

    dice = float(np.mean(dice_vals)) if dice_vals else 0.0
    iou  = float(np.mean(iou_vals))  if iou_vals  else 0.0
    return dice, iou, np.array(pred_cls), labels.cpu().numpy()


total_dice, total_iou, n = 0.0, 0.0, 0
all_pred, all_true = [], []

with torch.no_grad():
    for images, masks, labels in tqdm(test_loader, desc="Evaluating"):
        images = images.to(device)
        masks  = masks.to(device)
        pred   = model(images)

        d, iou, p_cls, t_cls = evaluate_batch(pred, masks, labels)
        total_dice += d; total_iou += iou
        all_pred.extend(p_cls); all_true.extend(t_cls)
        n += 1

mean_dice = total_dice / n
mean_iou  = total_iou  / n
acc   = accuracy_score(all_true, all_pred)
prec, rec, f1, _ = precision_recall_fscore_support(
    all_true, all_pred, average='binary', zero_division=0)
cm = confusion_matrix(all_true, all_pred)

print("\n" + "="*52)
print("      FINAL EVALUATION  (paper metric format)")
print("="*52)
print(f"  Dice Coefficient  : {mean_dice:.4f}")
print(f"  IoU Score         : {mean_iou:.4f}")
print(f"  Accuracy          : {acc*100:.2f}%")
print(f"  Precision         : {prec*100:.2f}%")
print(f"  Recall            : {rec*100:.2f}%")
print(f"  F1-Score          : {f1*100:.2f}%")
print("="*52)
print(f"  Confusion Matrix:")
print(f"    TP={cm[1,1]}  FN={cm[1,0]}")
print(f"    FP={cm[0,1]}  TN={cm[0,0]}")
print("="*52)


In [ ]:
# Cell 15: Visualisation — uses same thresholds as evaluation

model.eval()
images, masks, labels = next(iter(test_loader))

with torch.no_grad():
    pred_logits = model(images.to(device))
    pred_probs  = torch.sigmoid(pred_logits).cpu().numpy()

fig, axes = plt.subplots(3, 4, figsize=(16, 12))

for i in range(4):
    total_px   = pred_probs[i][0].size
    active_px  = int((pred_probs[i][0] > SIGMOID_THRESH).sum())
    pred_label = "Fractured" if active_px > PIXEL_RATIO * total_px else "Normal"
    confidence = (active_px / total_px) * 100
    true_label = "Fractured" if labels[i] == 1 else "Normal"
    match = "Match" if pred_label == true_label else "Miss"
    
    axes[0, i].imshow(images[i][0].numpy(), cmap='gray')
    axes[0, i].set_title(f"GT: {true_label}", fontsize=10)
    axes[0, i].axis('off')

    axes[1, i].imshow(masks[i][0].numpy(), cmap='bone')
    axes[1, i].set_title("Ground Truth Mask", fontsize=10)
    axes[1, i].axis('off')

    axes[2, i].imshow(pred_probs[i][0], cmap='hot', vmin=0, vmax=1)
    axes[2, i].set_title(
        f"{match} Pred: {pred_label}\n{active_px}px active ({confidence:.2f}%)",
        fontsize=9)
    axes[2, i].axis('off')

plt.suptitle("AWC Loss — Radillah et al. 2025  |  Fracture Segmentation",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
import os

# 1. Define the exact folder path
test_folder_path = "/kaggle/input/datasets/gowthambatthula/testimages/testImages"

if not os.path.exists(test_folder_path):
    print(f"⚠️ Folder not found at: {test_folder_path}")
else:
    # Get all image files in the directory
    image_files = sorted([f for f in os.listdir(test_folder_path) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.tif'))])
    
    if not image_files:
        print(f"❌ No images found in {test_folder_path}")
    else:
        print(f"Found {len(image_files)} images. Running inference...\n")
        
        # Setup plot: rows = number of images, cols = 3 (Input | Pred Mask | Red Overlay)
        fig, axes = plt.subplots(len(image_files), 3, figsize=(16, 5 * len(image_files)))
        
        # Ensure axes is 2D in case there is only 1 image
        if len(image_files) == 1:
            axes = np.expand_dims(axes, axis=0)
            
        model.eval()
        
        # Using the same thresholds from your evaluation cell (Cell 14)
        SIGMOID_THRESH = 0.40  # updated
        PIXEL_RATIO = 0.001  # updated 
        
        # Initialize the transform pipeline
        val_test_transforms = get_preprocessing_transform()
        
        for idx, img_name in enumerate(image_files):
            img_path = os.path.join(test_folder_path, img_name)
            
            # 2. Load the image strictly in Grayscale
            raw_image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            
            if raw_image is None:
                print(f"❌ Error loading image: {img_name}")
                continue
            raw_image = cv2.resize(raw_image, (512, 512))
            # 3. Preprocess using your existing pipeline
            dummy_mask = np.zeros_like(raw_image)
            augmented = val_test_transforms(image=raw_image, mask=dummy_mask)
            input_tensor = augmented['image'].unsqueeze(0).to(device)
            
            # 4. Model Inference
            with torch.no_grad():
                pred_mask_logits = model(input_tensor)
                # Convert logits to probability map (0.0 to 1.0)
                pred_mask_prob = torch.sigmoid(pred_mask_logits).cpu().numpy()[0][0]
                
            # 5. Calculate Classification Label
            total_px = pred_mask_prob.size
            mask_binary = pred_mask_prob > SIGMOID_THRESH
            active_px = mask_binary.sum()
            
            pred_label = "Fractured" if active_px > (PIXEL_RATIO * total_px) else "Normal"
            confidence = (active_px / total_px) * 100
            
            # 6. Visualize the Result
            preprocessed_image = input_tensor.cpu().numpy()[0][0]
            
            # --- OVERLAY CREATION ---
            # Normalize the image to [0, 1] to convert it to an RGB format
            img_min, img_max = preprocessed_image.min(), preprocessed_image.max()
            base_img_norm = (preprocessed_image - img_min) / (img_max - img_min + 1e-8)
            
            # Stack the 1-channel grayscale into 3-channel RGB
            overlay_img = np.stack([base_img_norm] * 3, axis=-1)
            
            # Apply Red color [1.0, 0.0, 0.0] to the masked areas (blending at 50% opacity)
            overlay_img[mask_binary] = overlay_img[mask_binary] * 0.5 + np.array([1.0, 0.0, 0.0]) * 0.5
            # ------------------------

            # Input Image (Left Column)
            axes[idx, 0].imshow(preprocessed_image, cmap='gray')
            axes[idx, 0].set_title(f"Input: {img_name}\nPred: {pred_label}")
            axes[idx, 0].axis('off')
            
            # Model Predicted Mask Output (Middle Column)
            axes[idx, 1].imshow(pred_mask_prob, cmap='hot', vmin=0, vmax=1)
            axes[idx, 1].set_title(f"Predicted Probability Map\nActive px: {active_px} ({confidence:.2f}%)")
            axes[idx, 1].axis('off')
            
            # Overlay Image (Right Column)
            axes[idx, 2].imshow(overlay_img)
            axes[idx, 2].set_title("Mask Overlay (Red)")
            axes[idx, 2].axis('off')
            
        plt.tight_layout()
        plt.show()